# Micro-benchmark de coste — dimensionar la campaña externa (CPU)

**Propósito:** convertir "me preocupa el tiempo" en horas concretas. Cronometra **un** candidato torch representativo por tarea **en CPU, sin tope**, y extrapola al grid completo × semillas × modelos para varios escenarios. Con el número delante se decide el reparto semillas/candidatos.

**Principio (no negociable):** el benchmark sirve para elegir *cuántas semillas/candidatos* caben en el presupuesto, **no** para recortar épocas a las redes. Si no cabe, se baja nº de semillas o se trocea la campaña, nunca se infra-entrena al competidor. Las redes paran por convergencia (early-stopping) o por un límite de épocas razonable, declarado y aplicado por igual.

**Objetivo de tiempo:** 1-2 h para la comparación externa completa.

In [1]:
import sys, time
from pathlib import Path
from itertools import product

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch

from rc_lab.runners.runner import resolve_task
from rc_lab.sequence_models.torch_models import fit_torch_sequence_model

# CPU forzado: la campaña final va en CPU para que los tiempos sean comparables entre familias.
DEVICE = "cpu"
torch.set_num_threads(torch.get_num_threads())  # usa los hilos por defecto del i9
print("device:", DEVICE, "| threads:", torch.get_num_threads())
SEED = 42

device: cpu | threads: 24


## 1. Configuración del benchmark

Cronometramos el candidato **más caro plausible** de cada modelo (hidden alto, sin tope) para tener una cota superior honesta del coste por candidato. Usamos el límite de épocas que irá a la campaña.

In [10]:
# Límite de épocas de la campaña (declarado, aplicado por igual). El sanity mostró
# convergencia típica en 250-350; 450 deja margen sin desperdiciar demasiado.
CAMPAIGN_MAX_EPOCHS = 800
CAMPAIGN_PATIENCE = 50

# Candidato representativo CARO (cota superior): hidden 128, L=1, sin tope.
def bench_cfg(bptt):
    return {
        "hidden_size": 128, "num_layers": 1, "learning_rate": 3e-3,
        "weight_decay": 0.0, "bptt_length": bptt,
        "max_epochs": CAMPAIGN_MAX_EPOCHS, "patience": CAMPAIGN_PATIENCE,
        "grad_clip": 1.0, "batch_size": 32, "window_stride": 50,
        "training_mode": "windowed", "normalize_inputs": True, "normalize_targets": True,
        # SIN max_train_seconds_per_run -> sin tope (igual que la campaña)
    }

# Tareas con su parametrización EXACTA de campaña (debe coincidir con los YAML finales).
TASKS = {
    "delay_recall": {
        "task_cfg": {"kmax": 100, "input_low": -1.0, "input_high": 1.0},
        "splits": {"n_train": 3000, "n_val": 1000, "n_test": 1500, "washout": 200},
        "bptt": 200, "metrics": ["memory_corr_total", "nmse"],
    },
    "narma10": {
        "task_cfg": {},
        "splits": {"n_train": 3000, "n_val": 1000, "n_test": 1500, "washout": 200},
        "bptt": 150, "metrics": ["nmse", "rmse"],
    },
    "mackey_glass": {
        "task_cfg": {"tau": 17, "dt": 0.1, "beta": 0.2, "gamma": 0.1, "n": 10,
                      "initial_history": "random_uniform", "history_low": 1.1, "history_high": 1.3,
                      "discard_transient": 1000, "sample_stride": 10, "prediction_horizon": 84},
        "splits": {"n_train": 3000, "n_val": 1000, "n_test": 1500, "washout": 200},
        "bptt": 200, "metrics": ["nmse", "rmse"],
    },
}
KINDS = ["torch_lstm", "torch_simple_rnn"]

## 2. Cronometrar un candidato por (tarea, modelo)

Mide tiempo real de un entrenamiento completo en CPU. También registra `best_epoch` para ver si el límite de épocas se está alcanzando (señal de que aún aprenden).

In [11]:
rows = []
for tname, spec in TASKS.items():
    task = resolve_task(tname, state_policy="reset", task_cfg=spec["task_cfg"])
    td = task.generate(**spec["splits"], seed=SEED)
    for kind in KINDS:
        cfg = bench_cfg(spec["bptt"])
        t0 = time.perf_counter()
        out = fit_torch_sequence_model(
            kind=kind, task_data=td, config_point=cfg, metrics=spec["metrics"],
            seed=SEED, device=DEVICE, evaluate_test=True,
            task_name=tname, task_cfg=spec["task_cfg"],
        )
        dt_s = time.perf_counter() - t0
        md = out["metadata"]
        rows.append({
            "task": tname, "kind": kind.replace("torch_", ""),
            "seconds": round(dt_s, 1), "epochs_ran": md.get("epochs_ran"),
            "best_epoch": md.get("best_epoch"), "status": md.get("status"),
            "hit_epoch_cap": md.get("epochs_ran") is not None and md.get("epochs_ran") >= CAMPAIGN_MAX_EPOCHS,
        })
        print(f"{tname:14s} {kind.replace('torch_',''):11s} {dt_s:7.1f}s  "
              f"epochs={md.get('epochs_ran')} best={md.get('best_epoch')} status={md.get('status')}")

bench = pd.DataFrame(rows)
bench

delay_recall   lstm           29.8s  epochs=194 best=144 status=ok
delay_recall   simple_rnn    166.1s  epochs=800 best=796 status=ok
narma10        lstm           24.7s  epochs=169 best=119 status=ok
narma10        simple_rnn     38.9s  epochs=244 best=194 status=ok
mackey_glass   lstm           75.8s  epochs=454 best=404 status=ok
mackey_glass   simple_rnn     74.5s  epochs=398 best=348 status=ok


,task,kind,seconds,epochs_ran,best_epoch,status,hit_epoch_cap
0,delay_recall,lstm,29.8,194,144,ok,False
1,delay_recall,simple_rnn,166.1,800,796,ok,True
2,narma10,lstm,24.7,169,119,ok,False
3,narma10,simple_rnn,38.9,244,194,ok,False
4,mackey_glass,lstm,75.8,454,404,ok,False
5,mackey_glass,simple_rnn,74.5,398,348,ok,False


In [7]:
# Diagnóstico: ¿el límite de épocas trunca el aprendizaje? Si hit_epoch_cap=True en muchas,
# subir CAMPAIGN_MAX_EPOCHS o aceptar el límite como restricción declarada (NUNCA recortar para favorecer ESN).
if bench["hit_epoch_cap"].any():
    print("AVISO: alguna config alcanza el tope de épocas con margen de aprendizaje:")
    print(bench[bench["hit_epoch_cap"]][["task","kind","best_epoch"]].to_string(index=False))
    print("-> Decisión honesta: subir el cap o declararlo como límite uniforme. No bajarlo selectivamente.")
else:
    print("OK: ninguna config representativa agota el tope de épocas; convergen antes.")

AVISO: alguna config alcanza el tope de épocas con margen de aprendizaje:
        task       kind  best_epoch
delay_recall simple_rnn         591
-> Decisión honesta: subir el cap o declararlo como límite uniforme. No bajarlo selectivamente.


## 3. Extrapolación al coste de la campaña

Coste externo ≈ Σ_tareas Σ_modelos (t_candidato × n_candidatos × n_semillas). El coste ESN es comparativamente despreciable (numpy, sin BPTT) — se ignora aquí como cota; si quieres, mídelo aparte. La cota es **superior** porque cronometramos el candidato más caro (hidden 128).

In [8]:
# Tiempo medio por candidato y tarea (promediando modelos; usar el peor por modelo si se prefiere conservador)
t_by_task = bench.groupby("task")["seconds"].mean().to_dict()
print("Segundos por candidato (medio entre lstm/rnn, cota alta por hidden=128):")
for k, v in t_by_task.items():
    print(f"  {k:14s} {v:7.1f}s")

N_MODELS_TORCH = len(KINDS)  # lstm + simple_rnn

def campaign_hours(n_candidates, n_seeds, tasks=None):
    tasks = tasks or list(t_by_task)
    total_s = sum(t_by_task[t] * N_MODELS_TORCH * n_candidates * n_seeds for t in tasks)
    return total_s / 3600.0

print("\n--- Escenarios (las 3 tareas, modelos torch; ESN aparte y despreciable) ---")
scenarios = [(12, 10), (12, 7), (12, 5), (6, 10), (6, 7), (6, 5)]
tab = []
for nc, ns in scenarios:
    h = campaign_hours(nc, ns)
    tab.append({"candidatos": nc, "semillas": ns, "horas_est": round(h, 2),
                "objetivo_1_2h": "OK" if h <= 2 else ("justo" if h <= 3 else "NO")})
print(pd.DataFrame(tab).to_string(index=False))

Segundos por candidato (medio entre lstm/rnn, cota alta por hidden=128):
  delay_recall      39.1s
  mackey_glass      39.0s
  narma10           16.6s

--- Escenarios (las 3 tareas, modelos torch; ESN aparte y despreciable) ---
 candidatos  semillas  horas_est objetivo_1_2h
         12        10       6.32            NO
         12         7       4.42            NO
         12         5       3.16            NO
          6        10       3.16            NO
          6         7       2.21         justo
          6         5       1.58            OK


In [9]:
# Vista por-tarea: permite trocear la campaña (correr tareas por separado) si el total no cabe.
print("Horas por tarea individual (modelos torch):")
for t in t_by_task:
    for nc, ns in [(12, 10), (6, 10), (6, 7)]:
        h = campaign_hours(nc, ns, tasks=[t])
        print(f"  {t:14s} cand={nc:2d} seeds={ns:2d} -> {h:.2f} h")
    print()

Horas por tarea individual (modelos torch):
  delay_recall   cand=12 seeds=10 -> 2.61 h
  delay_recall   cand= 6 seeds=10 -> 1.30 h
  delay_recall   cand= 6 seeds= 7 -> 0.91 h

  mackey_glass   cand=12 seeds=10 -> 2.60 h
  mackey_glass   cand= 6 seeds=10 -> 1.30 h
  mackey_glass   cand= 6 seeds= 7 -> 0.91 h

  narma10        cand=12 seeds=10 -> 1.11 h
  narma10        cand= 6 seeds=10 -> 0.55 h
  narma10        cand= 6 seeds= 7 -> 0.39 h



## 4. Decisión

Rellenar tras ejecutar:

1. **¿Qué escenario cabe en 1-2 h?** Mirar la tabla. Priorizar **semillas sobre candidatos** para la externa (la varianza entre semillas alimenta Friedman/Nemenyi de Fase 3). 5-7 semillas ya es mucho mejor que las 3 preliminares.
2. **¿El cap de épocas trunca aprendizaje?** Si sí, subirlo o declararlo uniforme — nunca recortarlo para favorecer la ESN.
3. **¿Trocear?** Si las 3 tareas juntas no caben, correr por tarea (cada una guarda sus resultados) es perfectamente válido y no afecta la comparación.
4. **Candidatos torch (rebalanceados según sanity):** `lr[1e-2,3e-3,1e-3] × hidden[64,128] × weight_decay[0,1e-5]` = 12, `bptt` fijo por tarea, `num_layers=1`. Si hay que bajar a 6, soltar `weight_decay`.
5. **Nº de candidatos ESN:** igualar al de las redes en la fase de comparación (fairness por candidatos). Recordar declarar en la memoria que el barrido baseline de región es adicional y solo lo tiene la ESN.